In [ ]:
import os
import dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains import LLMChain, SequentialChain

# 1. 加载环境变量
dotenv.load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY1")
os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL")

# 2. 初始化LLM
llm = ChatOpenAI(
    model='gpt-4o-mini',
    temperature=0.8,
    max_tokens=100  # 适当增加，避免输出被截断
)

# 3. 第一个Chain：生成产品卖点
prompt1 = ChatPromptTemplate.from_messages([
    ("system", "你是产品分析师，用1句话提炼产品核心卖点，突出差异化。"),
    ("human", "产品：{product_name}")
])
sell_point_chain = LLMChain(
    llm=llm,
    prompt=prompt1,
    output_key="sell_point"
)

# 4. 第二个Chain：生成营销文案
prompt2 = ChatPromptTemplate.from_messages([
    ("system", "你是文案师，用卖点写一句简短有力的营销文案，适合社交媒体传播。"),
    ("human", "产品卖点：{sell_point}")
])
copywriting_chain = LLMChain(
    llm=llm,
    prompt=prompt2,
    output_key="marketing_copy"
)

# 5. 组合顺序链（无任何Memory配置）
sequential_chain = SequentialChain(
    chains=[sell_point_chain, copywriting_chain],
    input_variables=["product_name"],
    output_variables=["sell_point", "marketing_copy"],
    verbose=True  # 可选：打印执行过程，方便调试
)

# 6. 运行（输入产品名，输出结果）
result = sequential_chain.invoke({"product_name": "便携折叠无线充电宝"})  # 推荐用invoke而非run，更符合新版API

# 7. 打印结果
print("="*50)
print(f"产品核心卖点：{result['sell_point']}")
print(f"营销文案：{result['marketing_copy']}")